In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
parsed_args = RayTracing.parse_commandline()
    
parsed_args["scene-number"] = 1

1

In [3]:
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

Random.TaskLocalRNG()

In [4]:
I, scene = RayTracing.build_scene(parsed_args)
wbounds = RayTracing.world_bounds(scene.b)


There are 91 objects in the scene, building BVH
  0.027071 seconds (86.00 k allocations: 6.122 MiB, 96.63% compilation time)
Done building BVH
Using 5 samples per pixel
There are 42 lights in the scene


Main.RayTracing.Bounds3([-978.8225099390856, -0.0001, -992.9646455628165], [300.0, 265.0, 300.0])

In [5]:
N = 4
N_voxels = N^3

64

In [6]:
function calc_voxel_size(wbounds, N_voxels, max_recursion=100)
    size = floor((prod(wbounds.pMax - wbounds.pMin) / N_voxels)^(1/3))
    N = prod(floor.((wbounds.pMax - wbounds.pMin)/size) .+1)
    depth = 1
    while (N > N_voxels) & (depth <= max_recursion)
        depth += 1
        N = prod(floor.((wbounds.pMax - wbounds.pMin)/(size+depth)) .+1)
    end
    @assert prod(floor.((wbounds.pMax - wbounds.pMin)/(size+depth-1)) .+ 1) >= N_voxels
    return size + depth - 1
end

calc_voxel_size (generic function with 2 methods)

In [7]:
size = calc_voxel_size(wbounds, N_voxels)

255.0

In [18]:
X, Y, Z = Int64.(floor.((wbounds.pMax - wbounds.pMin)/size) .+1)

3-element StaticArraysCore.SVector{3, Int64} with indices SOneTo(3):
 6
 2
 6

In [19]:
# VoxelStruct members
# X, Y, Z, size, wbounds, 
# Dict{Tuple{Int64, Int64, Int64}, Int64} integer coords
# key: tuple integers represent the bottom left coord
# value: light index

In [33]:
function sample_point_on_light(
    light::RayTracing.Light, 
    voxel_isect::RayTracing.Interaction, 
    u::RayTracing.Pnt2, 
    scene::RayTracing.BVHAccel
)::RayTracing.Maybe{RayTracing.Pnt3}
    if light.flags == RayTracing.LightArea
        p, n, light_pdf = RayTracing.sample(light.shape, u)
        vis = RayTracing.VisibilityTester(voxel_isect, RayTracing.Interaction(p, 0.0, n))
        check = RayTracing.unoccluded(vis, scene)
        L = check * light.Lemit / RayTracing.distance(vis.p0.p, vis.p1.p)
    elseif light.flags == RayTracing.LightInfinite 
        uv, map_pdf = RayTracing.sample_continuous(light.distribution, uvu)
        theta = uv.y * pi
        phi = uv.x * 2 * pi
        cos_theta = RayTracing.cos(theta)
        sin_theta = RayTracing.sin(theta)
        sin_phi = RayTracing.sin(phi)
        cos_phi = RayTracing.cos(phi)
        wi = il.light_to_world(RayTracing.Vec3(sin_theta * cos_phi, sin_theta * sin_phi, cos_theta))
        vis = RayTracing.VisibilityTester(
            interaction,
            RayTracing.Interaction(
                voxel_isect.p + wi .* 2 * light.world_radius, 
                voxel_isect.t, 
                RayTracing.Nml3(0, 0, 0), 
                RayTracing.Vec3(0, 0, 0),
                RayTracing.MediumInterface(light.medium)
            )
        )
        check = RayTracing.unoccluded(vis, scene)
        radiance = RayTracing.lookup(il.Lmap, uv)
        radiance = RayTracing.spectrum_from_RGB(radiance.a, radiance.b, radiance.c, Illuminant)
        L = vis * radiance / RayTracing.distance(vis.p0.p, vis.p1.p)
    else
        @assert false, "No VoxelLightDistribution for LightDeltaDirection or LightDeltaPosition yet"
    end

    return L
end

sample_point_on_light (generic function with 1 method)

In [34]:
function create_voxels_distribution(
    scene::RayTracing.Scene,
    sampler::RayTracing.AbstractSampler,
    voxel_x_dim::Int64,
    voxel_y_dim::Int64,
    voxel_z_dim::Int64,
    n_shadow_rays::Int64
)::Dict{Tuple{Int64, Int64, Int64}, RayTracing.Distribution1D}
    voxels = Dict{Tuple{Int64, Int64, Int64}, RayTracing.Distribution1D}()
    sampler = RayTracing.IndependentSampler()
    for x in 1:voxel_x_dim
        for y in 1:voxel_y_dim
            for z in 1:voxel_z_dim
                light_samples = zeros(Float64, length(scene.lights))

                # TODO: how to cull voxels functionally outside of the scene
                lcorner = RayTracing.Pnt3(x-1, y-1, z-1) .* size
                center = lcorner .+ size/2
                # TODO: account for mediums
                voxel_isect = RayTracing.Interaction()
                voxel_isect.p = center

                for idx in eachindex(scene.lights)
                    LL = RayTracing.spectrum_from_float(0.0)
                    for n in n_shadow_rays
                        u = RayTracing.get_2D!(sampler)
                        LL += sample_point_on_light(scene.lights[idx], voxel_isect, u, scene.b)
                    end
                    light_samples[idx] = RayTracing.y_spectrum(LL) / n_shadow_rays
                end
                voxels[(x,y,z)] = RayTracing.Distribution1D(light_samples)
            end
        end
    end
    return voxels
end

create_voxels_distribution (generic function with 1 method)

In [35]:
S = RayTracing.IndependentSampler()

Main.RayTracing.IndependentSampler()

In [36]:
voxel_dist = create_voxels_distribution(scene, S, X, Y, Z, 3)

Dict{Tuple{Int64, Int64, Int64}, Main.RayTracing.Distribution1D} with 72 entries:
  (1, 2, 3) => Distribution1D([0.002059, 0.00206413, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  (3, 1, 3) => Distribution1D([0.00180159, 0.00178185, 0.00194846, 0.00194193, …
  (5, 1, 6) => Distribution1D([0.000868846, 0.000869225, 0.000938399, 0.0009370…
  (3, 2, 4) => Distribution1D([0.00138073, 0.00137301, 0.00150969, 0.0015091, 0…
  (3, 2, 3) => Distribution1D([0.00169007, 0.00166829, 0.0, 0.0, 0.0, 0.0, 0.0,…
  (5, 2, 6) => Distribution1D([0.000852636, 0.000852374, 0.000928268, 0.0009325…
  (2, 1, 1) => Distribution1D([0.00420031, 0.00429381, 0.00484077, 0.00483382, …
  (6, 1, 6) => Distribution1D([0.000795177, 0.000794692, 0.000860631, 0.0008551…
  (5, 1, 4) => Distribution1D([0.00110267, 0.00109148, 0.00118083, 0.0011782, 0…
  (2, 2, 1) => Distribution1D([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0…
  (6, 2, 6) => Distribution1D([0.000786529, 0.000783433, 0.00085361, 0.00085029…
  (5, 1, 3) => Distribution

In [46]:
wbounds = RayTracing.world_bounds(scene.b)
p = RayTracing.centroid(RayTracing.world_bounds(scene.b))
p = wbounds.pMin
p = wbounds.pMax

distance_from_pmin = Int64.(floor.((p - wbounds.pMin)/size) .+ 1)
voxel_dist[(distance_from_pmin.x, distance_from_pmin.y, distance_from_pmin.z)]

Main.RayTracing.Distribution1D([0.00078652883262946, 0.0007834325856000408, 0.0008536104909778803, 0.0008502910559213421, 0.000241959718359937, 0.0002418826836465262, 0.0, 0.0, 0.0, 0.00018220570084319363  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0005854708361353382, 0.0, 0.0, 0.0], [0.0, 0.11744941745841364, 0.23443648389442814, 0.3619029552335328, 0.4888737477144599, 0.5250046901425608, 0.5611241292641854, 0.5611241292641854, 0.5611241292641854, 0.5611241292641854  …  0.912573823373897, 0.912573823373897, 0.912573823373897, 0.912573823373897, 0.912573823373897, 0.912573823373897, 1.0, 1.0, 1.0, 1.0], 0.00015944631631739585)